# Demo Notebook

This is a demo notebook to show you how to load the data from Google BigQuery into a pandas dataframe and then use the data to construct a simple visualization.

In [ ]:
import sys
print(sys.executable)

In [ ]:
import plotly.express as px
import numpy as np
import json

In [ ]:
# read bigquery data into pandas dataframe
import pandas as pd

df = pd.read_gbq(
    """
    SELECT  *
    FROM `jr-data-training.cafe.cafe-sales`
    --- LIMIT 10
  """,
    project_id="jr-data-training",
    location="australia-southeast1",
)

df.head()

# EDA

## Initial EDA

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.nunique()

- From the above overlook status, we can see that **1353** columns are empty in **date_paid**.
- **Status** is considered as category column, as there are only 5 different values.
- The **item** column value is a JSON array, it need to be divided into more specific parts.

In [ ]:
df['status'].unique()

## Revenue Trend

### Monthly revenue trend analysis

In [ ]:
df['date_created']
df['month'] = df['date_created'].dt.to_period('M')
monthly_sales = df.groupby('month')['total'].sum() / 100

fig = px.line(x=monthly_sales.index.astype(str), y=monthly_sales.values, labels={'x':'Month', 'y':'Total Sales($AUD)'}, title='Monthly Sales')
fig.show()

The trend of the monthly sales are overall fluctuated. It increased slightly from May 2019 to around Sep 2020, then the sales figures fluctuate but generally maintain a higher level compared to the initial period.

- The peak is observed in Sep 2021, with a total of 13.26K(AUD).
- Keep in mind that there are spikes around each end of the year (Oct to Dec).
- A significant drop is observed in Mar 2024.


### Average order value examination

In [ ]:
AOV = df.groupby('month')['total'].mean() / 100

fig = px.line(x=AOV.index.astype(str), y=AOV.values, labels={'x':'Month', 'y':'Average Sales($AUD)'}, title='Monthly Average Sales')
fig.show()

The metric illustrates a relatively stable values with occasional peak from May 2019 to Mar 2020. Then fluctuation occurs in around Apr 2020 to Jun 2021. After reaching the highest peak, the overall trend appears to be downward. 

- The highest peak is in Sep 2021, with AOV 14.68(AUD).
- The lowest point is in May 2023, AOV 9.21(AUD).

### Identification of peak revenue hours.

In [ ]:
df['time'] = df['date_created'].dt.strftime('%H')
hour_sales = df[(df["status"] == 2)].groupby('time')['total'].sum() / 100

fig = px.line(x=hour_sales.index.astype(str), y=hour_sales.values, labels={'x':'Hour', 'y':'Total Sales($AUD)'}, title='Hourly Sales')
fig.show()

Obviously, the peak revenue hour is at 8:00A.M, with a total of 86.97K revenue.